In [22]:
import numpy as np 
import matplotlib.pyplot as plt 
import pickle
import pyvisa
import time 
from datetime import datetime  
import nysg_tools as ny
import os
import pickle
import glob as glob

In [103]:
rm = pyvisa.ResourceManager()

# rm=pyvisa.ResourceManager('@py')
#Acá abajo se van a listar los componentes q esten conectados a la PC

In [104]:
print(rm.list_resources('USB?*::INSTR'))

('USB0::0x0699::0x0363::C065092::INSTR', 'USB0::0x0699::0x0346::C036493::INSTR')


In [106]:

osci = rm.open_resource('USB0::0x0699::0x0363::C065092::INSTR') #Osciloscopio
fungen = rm.open_resource('USB0::0x0699::0x0346::C036493::INSTR') #Generador de funciones

In [18]:
def get_osci(additional_info = False, plot=False):
    """
    Adquiere las waveforms de los canales CH1 y CH2 del osciloscopio.

    Parameters
    ----------
    additional_info : bool, opcional
        Si es True, devuelve un diccionario con información adicional de
        calibración y datos crudos (útil para debugging o reprocesamiento).
        Por defecto es False.

    plot : bool, opcional
        Si es True, grafica ambas señales (voltaje vs tiempo).
        Por defecto es False.

    Returns
    -------
    tiempo : np.ndarray
        Array de tiempos correspondiente a las muestras.

    data1v : np.ndarray
        Voltajes del canal 1 (V).

    data2v : np.ndarray
        Voltajes del canal 2 (V).

    extra : dict, opcional
        Solo si additional_info=True. Contiene:
            - yze1, ymu1, yoff1 : parámetros de escala CH1
            - yze2, ymu2, yoff2 : parámetros de escala CH2
            - datach1raw, datach2raw : datos crudos del ADC

    Notas
    -----
    - Se asume que el osciloscopio ya está conectado y asignado a `osci`.
    - Conversión a voltaje:
        V = (raw - yoff) * ymu + yze
    - Eje temporal:
        t = xze + n * xin

    Ejemplos de uso
    ----------------
    >>> t, v1, v2 = get_osci()
    >>> t, v1, v2, extra = get_osci(additional_info=True)
    >>> get_osci(plot=True)
    """
    osci.write("DAT:SOU CH1")
    xze, xin = osci.query_ascii_values('WFMPRE:XZE?;XIN?', separator=';')
    yze1, ymu1, yoff1 = osci.query_ascii_values('WFMPRE:YZE?;YMU?;YOFF?;', separator=';')
    osci.write("DAT:SOU CH2")

    yze2, ymu2, yoff2 = osci.query_ascii_values('WFMPRE:YZE?;YMU?;YOFF?;', separator=';')

    osci.write('DAT:ENC RPB')
    osci.write('DAT:WID 1')

    osci.write("DAT:SOU CH1")
    data1 = osci.query_binary_values('CURV?', datatype='B', container=np.array)

    osci.write("DAT:SOU CH2")
    data2 = osci.query_binary_values('CURV?', datatype='B', container=np.array)

    tiempo = xze + np.arange(len(data1)) * xin

    data1v = (data1 - yoff1) * ymu1 + yze1
    data2v = (data2 - yoff2) * ymu2 + yze2

    if plot:
        plt.plot(tiempo,data1v)
        plt.plot(tiempo,data2v)

    if additional_info:
        # extra = {"yze1":yze1,"ymu1":ymu1,"yoff1":yoff1,"yze2":yze2,"ymu2":ymu2,"yoff2":yoff2,"datach1raw":data1,"datach2raw":data2}
        extra = {"sca1": osci.query("CH1:SCA?"), "sca2": osci.query("CH2:SCA?") }
        return tiempo,data1v,data2v, extra

    return tiempo,data1v,data2v

In [19]:
def setup_save_folder(): #Auto hace una carpeta 
    # Creates a folder like "14h_30m_45s"
    folder_name = datetime.now().strftime("%Hh_%Mm_%Ss")
    os.makedirs(folder_name, exist_ok=True)
    return folder_name

In [ ]:
segments= [
    
    (150140,150170, 0.5 ),
    (248275,248295, 0.7  ),
    (427100,427550, 1.2 ) ,
    (558640,560670,20), 
    (648850, 650580, 30)

                      ]
freqs = []
for start, stop, step in segments:
    segment_array = np.arange(start, stop, step)
    freqs.append(segment_array)

freq_array = np.concatenate(freqs)

freq_array = np.unique(np.round(freq_array, decimals=3))

In [166]:
len(freq_array)

269

In [149]:
vpp_ch1 = osci.query("MEASU:MEAS1:VALUE?")
vpp_ch2 = osci.query("MEASU:MEAS2:VALUE?")

In [150]:
vpp_ch2

'1.340000033E-1\n'

In [151]:
vpp_ch1

'2.2000000477E0\n'

In [146]:
osci.write("MEASU:MEAS1:SOURCE CH1")
time.sleep(0.1)

osci.write("MEASU:MEAS1:TYPE PK2PK")
time.sleep(0.1)
osci.write("MEASU:MEAS2:SOURCE CH2")
time.sleep(0.1)

osci.write("MEASU:MEAS2:TYPE PK2PK")



24

In [172]:
total_points = len(freq_array)

save_dir = setup_save_folder() # crea carpeta 

print(f"🧈 Starting sweep... Total points: {total_points}")

for i, freq in enumerate(freq_array):

    fungen.write(f"SOURCE1:FREQ {freq}")

    time.sleep(0.2)
    vpp1 = osci.query("MEASU:MEAS1:VALUE?")
    time.sleep(0.2)
    vpp2 = osci.query("MEASU:MEAS2:VALUE?")

    # Filename format: 0000_50000.000Hz.pkl so they sort perfectly in your file explorer
    filename = os.path.join(save_dir, f"{i:04d}_{freq:.3f}Hz.pkl")

    # Save the data as pickle
    data_to_save = {
        # 't': t,
        'v1': vpp1,
        'v2': vpp2,
        # 'scale': extra  # metadata from extra (scale information)
    }
    
    with open(filename, 'wb') as f:
        pickle.dump(data_to_save, f)

    # 5. Print progress
    print(f"[{i+1}/{total_points}] Saved {filename} 🧈")

🧈 Starting sweep... Total points: 404
[1/404] Saved 13h_55m_38s\0000_248275.000Hz.pkl 🧈
[2/404] Saved 13h_55m_38s\0001_248275.700Hz.pkl 🧈
[3/404] Saved 13h_55m_38s\0002_248276.400Hz.pkl 🧈
[4/404] Saved 13h_55m_38s\0003_248277.100Hz.pkl 🧈
[5/404] Saved 13h_55m_38s\0004_248277.800Hz.pkl 🧈
[6/404] Saved 13h_55m_38s\0005_248278.500Hz.pkl 🧈
[7/404] Saved 13h_55m_38s\0006_248279.200Hz.pkl 🧈
[8/404] Saved 13h_55m_38s\0007_248279.900Hz.pkl 🧈
[9/404] Saved 13h_55m_38s\0008_248280.600Hz.pkl 🧈
[10/404] Saved 13h_55m_38s\0009_248281.300Hz.pkl 🧈
[11/404] Saved 13h_55m_38s\0010_248282.000Hz.pkl 🧈
[12/404] Saved 13h_55m_38s\0011_248282.700Hz.pkl 🧈
[13/404] Saved 13h_55m_38s\0012_248283.400Hz.pkl 🧈
[14/404] Saved 13h_55m_38s\0013_248284.100Hz.pkl 🧈
[15/404] Saved 13h_55m_38s\0014_248284.800Hz.pkl 🧈
[16/404] Saved 13h_55m_38s\0015_248285.500Hz.pkl 🧈
[17/404] Saved 13h_55m_38s\0016_248286.200Hz.pkl 🧈
[18/404] Saved 13h_55m_38s\0017_248286.900Hz.pkl 🧈
[19/404] Saved 13h_55m_38s\0018_248287.600Hz.pkl 🧈
[2